# Score a recording with the trained ensemble

Load the four seed-42 detectors for one machine type and compare their combined anomaly score with the healthy-training cutoff for its known ID. Use a ten-second, 16 kHz WAV and microphone channel 1. The study covers fans, pumps, sliders and valves with IDs 00, 02, 04 and 06.

Extract [the final ensemble](https://github.com/AnAvenger9/mimii-acoustic-anomaly-detection/releases/download/models-v1/final-ensemble.zip) under `checkpoints/final-ensemble/` in the repository. This notebook uses PyTorch, torchvision, XGBoost, librosa and soundfile. See the [installation steps](../REPRODUCE.md#run-the-trained-ensemble).

In [ ]:
import hashlib
import json
import os
from pathlib import Path

import numpy as np
import torch
import xgboost as xgb

root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "notebooks/00_runtime.ipynb").is_file())
os.chdir(root)
get_ipython().run_line_magic("run", '"notebooks/00_runtime.ipynb"')
from mimii.audio import load_audio
from mimii.config import PreprocessConfig
from mimii.features import log_mel_spectrogram, normalize
from mimii.large_acoustic_cnn import LargeAcousticResidualStatsCNN
from mimii.train import known_id_anomaly_scores
from mimii.transfer_models import build_transfer

## Choose the recording and its installed machine ID

The example path is a fan clip from the original 0 dB release. Set these inputs for your WAV. Each machine type has its own fitted models and normalization. CPU inference is sufficient.

In [ ]:
bundle_dir = Path("checkpoints/final-ensemble")
wav_path = Path("data/fan_0db/fan/id_00/abnormal/00000000.wav")
machine = "fan"
machine_id = "id_00"

## Extract log-mel features and score the four members

Each CNN returns logits for four machine IDs. XGBoost returns four raw margins from 256 band statistics. The same numerically stable negative log probability of the known ID converts both output types to anomaly scores.

In [ ]:
def score_wav(wav_path, machine, machine_id, bundle_dir):
    manifest = json.loads((bundle_dir / "manifest.json").read_text())
    machine_config = manifest["machines"][machine]
    calibration = machine_config["ids"][machine_id]
    target = torch.tensor([manifest["machine_ids"][machine_id]])
    frontend = PreprocessConfig(**manifest["frontend"])
    waveform, _ = load_audio(wav_path, frontend)
    with np.load(bundle_dir / machine_config["normalizer"], allow_pickle=False) as stats:
        x = normalize(log_mel_spectrogram(waveform, frontend), stats["mean"], stats["std"])

    members = {m["model"]: m for m in manifest["models"] if m["machine"] == machine and m["seed"] == 42}
    raw_scores = {}
    standardized = []
    for name in manifest["fusion_members"]:
        entry = members[name]
        path = bundle_dir / entry["path"]
        if hashlib.sha256(path.read_bytes()).hexdigest() != entry["sha256"]:
            raise ValueError(f"Weight checksum mismatch for {path.name}")
        if name == "xgboost":
            model = xgb.Booster()
            model.load_model(path)
            summary = np.concatenate([x.mean(axis=1), x.std(axis=1)])
            margins = model.predict(xgb.DMatrix(summary[None, :]), output_margin=True)
            logits = torch.from_numpy(margins)
        else:
            model = LargeAcousticResidualStatsCNN() if name == "wide" else build_transfer(name, pretrained=False)
            model.load_state_dict(torch.load(path, weights_only=True, map_location="cpu"))
            with torch.inference_mode():
                logits = model.eval()(torch.from_numpy(x)[None, None])
        score = known_id_anomaly_scores(logits, target).item()
        raw_scores[name] = score
        moments = calibration["log_score_stats"][name]
        standardized.append((np.log(max(score, np.finfo(float).tiny)) - moments["mean"]) / moments["std"])

    fused = float(np.mean(standardized))
    cutoff = calibration["fused_cutoff"]
    return raw_scores, fused, cutoff, fused > cutoff

## Read the decision

The cutoff comes from the 95th percentile of fused healthy training scores for this type and ID. An alarm indicates that a clip merits inspection. The detector was evaluated on known IDs and does not assign a physical cause.

In [ ]:
member_scores, score, cutoff, alarm = score_wav(wav_path, machine, machine_id, bundle_dir)
print(f"{machine} / {machine_id}  {wav_path.name}")
print(f"Fused score {score:.4f} | cutoff {cutoff:.4f} | {'ALARM' if alarm else 'below threshold'}")

On the example fan recording, the published ensemble gives a fused score of about 0.551 against a cutoff of 1.054. MIMII labels this clip abnormal, so it is one of the missed fan faults in the report. Small floating-point differences can occur across hardware.